# 3. ResNet18-based Cell Cycle Classifier Training Script

This script trains a deep learning model (ResNet-18) to classify single-cell images into multiple substages
of the cell cycle (e.g., Early S, Mid S, G1G2, etc.).
----------------------------------------------------------------------
Final Project Folder Structure:
codes/
├── train.py                 # Main training script
├── model.py                # ResNet18 classifier definition
├── data_utils.py           # Data loading and splitting
├── train_utils.py          # Training loop, evaluation, plotting
----------------------------------------------------------------------
Variables:

# File paths and dataset structure
- base_path: str- The root folder that contains all patch subfolders (e.g., 'patches-esc-edu/')
- dataset_subfolders: List[str]- Names of subdirectories to load training data from

# Model saving paths
- save_path: File path to save the best model (.pth)
- csv_path:  File path to save epoch-wise training logs (.csv)

# Training hyperparameters
- batch_size:     Number of samples per batch for training
- initial_epochs: Number of initial training epochs before prompting for extension
- step_epochs:    Number of additional epochs for each user-confirmed training extension
- lr: float - Initial learning rate for the optimizer
- patience:       Number of epochs without improvement before triggering early stopping

# System & transformation
- device: 'cuda' if GPU is available, else 'cpu'
- transform: torchvision.transforms.Compose
    Image augmentation pipeline including flip, jitter, affine transform, resizing, and normalization

# Training objects (from data_utils.py)
- train_loader / val_loader / test_loader: Dataloaders for training, validation, and testing subsets
- class_to_idx: dict - Mapping from class name to integer label

# Model
- model: ResNetClassifier
    A modified ResNet-18 classifier with the final fully connected layer replaced to match `num_classes`
----------------------------------------------------------------------
Output:
- best_model_demo.pth: The model with the highest test accuracy during training
- training_log_demo.csv: CSV file logging epoch, train/val loss, test accuracy, and learning rate
- Graphs: 
    • Epoch loss curve
    • Batch loss curve
    • Test accuracy curve
    • Confusion matrix
    • Precision-recall curve per class

structure:
-input_folder
-output_root
    -output_folder 
        -ori_img_folder (nd2, tif)
            -segmentation(mask.png/tif)
                -vis(png)
            -results(csv)
            -eightbit(tif)
            -opencv(tif)
        -channel_folder (png, json)
    -patches_root
        -phase_folders (5 classes)
    -resnet_model
    -resnet_csv


In [ ]:
from model import ResNetClassifier
from data_utils import load_data
from train_utils import train_model_input
import torchvision.transforms as transforms
import torch
import os
from pathlib import Path


In [ ]:

if __name__ == "__main__":
    base_path = "/Users/josefin/PhD/weihua_UHRF1_CellCycle"

    dataset_subfolders = [
        'cell_annotations',
        # 'patches_edu'
        # In the future, you can also add 'patches_xyz'
    ]
    save_path = Path(base_path) / "best_model.pth"
    csv_path = Path(base_path) / "training_log.csv"


    batch_size = 10
    num_classes = 5
    device = torch.device('mps') 
    initial_epochs = 50
    step_epochs = 20
    lr = 0.001
    patience = 20

    transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(5),
        transforms.ColorJitter(brightness=0.3, contrast=0.3),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.Resize((224, 224)),
        transforms.ToTensor()
    ])

    train_loader, val_loader, test_loader, class_to_idx = load_data(base_path, dataset_subfolders, batch_size, transform)

    model = ResNetClassifier(num_classes=num_classes)
    state_dict = torch.load("/Users/josefin/PhD/weihua_UHRF1_CellCycle/models/cellcycle_classification/resnet18_cellcycle.pth", map_location=device)

    # Add "model." prefix to all keys
    state_dict = {"model." + k: v for k, v in state_dict.items()}

    model.load_state_dict(state_dict)


    all_preds, all_labels = train_model_input(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        device=device,
        initial_epochs=initial_epochs,
        step_epochs=step_epochs,
        lr=lr,
        patience=patience,
        save_path=save_path,
        csv_path=csv_path
    )


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
# new labels order
label_order = [1, 0, 2, 3, 4]
class_names = ["G1G2", "early", "mid", "late", "ambiguous"]


cm = confusion_matrix(all_labels, all_preds, labels=label_order, normalize='true')

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap="Blues")

plt.title("Confusion Matrix (normalized)")
plt.grid(False)
plt.savefig("/Users/josefin/PhD/weihua_UHRF1_CellCycle/models/cellcycle_classification/confusion_matrix.pdf", dpi=300, bbox_inches='tight')
plt.show()